# Scanner Dev Comparison

Compare two scanner runs from the `scanner_dev/ground_truth/` staging workflow side by side.

Each scanner config supplies a `label` (used in plot titles / table headings), a `scan_results_path` (a `scan_id=*` directory produced by `scout scan scout.yaml`) and a `scanner_key`. Provenance, validation CSV and target rules are shared across both scanners so the comparison is apples-to-apples. The provenance CSV supplies a `model` column for each transcript, so descriptive plots and per-eval metrics are split by `(benchmark, method, model)`.

**Plots** are laid out as a single row of two subplots (one per scanner). **Tables** are kept separate and rendered under a heading identifying the scanner.

**Target rules** are keyed by eval-file basename. Each rule is one of:
- `{"mode": "validation"}` — look up the per-transcript target in the merged validation CSV (e.g. human t5 labels)
- `{"mode": "uniform", "positive_rate": 1.0}` — assume every transcript is a violation (e.g. synthetic contamination / web-search runs)
- `{"mode": "uniform", "positive_rate": 0.0}` — assume no violations

Eval files not listed in `TARGET_RULES` still appear in descriptive plots but are skipped when computing performance metrics.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Reuse the shared loader from the main analysis package
_ANALYSIS_DIR = Path("../analysis").resolve()
if str(_ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(_ANALYSIS_DIR))

from scan_utils import load_scan_results  # noqa: E402

In [ ]:
import sys
from pathlib import Path

# Point this import at whichever scanner subdirectory you want to evaluate.
_CONFIG_DIR = Path("./ground_truth").resolve()

if str(_CONFIG_DIR) not in sys.path:
    sys.path.insert(0, str(_CONFIG_DIR))

from config import (  # noqa: E402
    EXCLUDE_EVAL_FILES,
    INCLUDE_EVAL_FILES,
    PROVENANCE_CSV,
    TARGET_RULES,
    VALIDATION_CSV,
    SCANNERS,
    VIOLATION_THRESHOLD,
)

## Load Data

For each configured scanner, loads the scan parquet, attaches the `eval_file` basename to each row, merges manifest metadata from the provenance CSV, and produces a `comparison` dataframe with resolved targets.

All per-scanner state is collected into a list of dicts (`scanner_runs`) that is reused by every cell below.

In [ ]:
# Shared inputs (dataset-level, identical for both scanners).
provenance = pd.read_csv(PROVENANCE_CSV)
provenance["eval_file"] = provenance["staged_eval_log_path"].apply(
    lambda p: Path(p).name if isinstance(p, str) else None
)
_meta_cols_all = ["eval_file", "Eval", "method", "model", "samples", "expected_v_rate",
                  "validation_path", "include_in_validation"]
_meta_cols = [c for c in _meta_cols_all if c in provenance.columns]
provenance_meta = provenance[_meta_cols].drop_duplicates(subset=["eval_file"])

validation = pd.read_csv(VALIDATION_CSV)
validation["target_num"] = pd.to_numeric(validation["target"], errors="coerce")
validated_ids = set(validation["id"].dropna())
validation_lookup = validation.set_index("id")["target_num"]

# Benchmark group: collapses eval files that belong to the same benchmark.
# BENCHMARK_ALIASES merges task_set variants (e.g. the "_mini" subset maps
# back onto its parent benchmark).
BENCHMARK_ALIASES = {
    "swe_bench_verified_mini": "swe_bench",
}


def _benchmark_group(row: pd.Series) -> str | None:
    ts = row.get("transcript_task_set")
    if isinstance(ts, str) and ts:
        return BENCHMARK_ALIASES.get(ts, ts)
    ev = row.get("Eval")
    return str(ev) if pd.notna(ev) else None


def _load_scanner(cfg: dict) -> dict:
    all_scans = load_scan_results(cfg["scan_results_path"])
    scans = all_scans[all_scans["scanner_key"] == cfg["scanner_key"]].copy()
    if scans.empty:
        available = sorted(all_scans["scanner_key"].dropna().unique())
        raise ValueError(
            f"[{cfg['label']}] Scanner key '{cfg['scanner_key']}' not found. "
            f"Available: {available}"
        )

    scans["eval_file"] = scans["transcript_source_uri"].apply(
        lambda uri: Path(uri).name if isinstance(uri, str) else None
    )
    scans = scans.merge(provenance_meta, on="eval_file", how="left")
    scans["benchmark"] = scans.apply(_benchmark_group, axis=1)

    if INCLUDE_EVAL_FILES:
        scans = scans[scans["eval_file"].isin(INCLUDE_EVAL_FILES)]
    if EXCLUDE_EVAL_FILES:
        scans = scans[~scans["eval_file"].isin(EXCLUDE_EVAL_FILES)]

    return {
        "label": cfg["label"],
        "scanner_key": cfg["scanner_key"],
        "scan_results_path": cfg["scan_results_path"],
        "scans": scans,
    }


scanner_runs = [_load_scanner(cfg) for cfg in SCANNERS]

for run in scanner_runs:
    scans = run["scans"]
    print(f"[{run['label']}] scanner_key={run['scanner_key']}")
    print(f"  Scan rows: {len(scans):,}")
    print(f"  Unique transcripts: {scans['transcript_id'].nunique():,}")
    print(f"  Unique eval files in scan: {scans['eval_file'].nunique():,}")
    print(f"  Unique benchmarks in scan: {scans['benchmark'].nunique():,}")
    if "model" in scans.columns:
        print(f"  Unique models in scan: {scans['model'].nunique(dropna=True):,}")

print(f"\nValidation CSV entries: {len(validation):,}")

In [ ]:
def _format_rule(rule: dict | None) -> str:
    if rule is None:
        return "—"
    mode = rule.get("mode")
    if mode == "validation":
        return "validation"
    if mode == "uniform":
        return f"uniform::{rule.get('positive_rate')}"
    return str(rule)


def _short_label(row: pd.Series) -> str:
    """Multi-line plot label: Eval / method / model."""
    eval_name = row.get("Eval") or row.get("transcript_task_set") or row["eval_file"]
    parts = [str(eval_name)]
    method = row.get("method")
    if method is not None and pd.notna(method):
        parts.append(str(method))
    model = row.get("model")
    if model is not None and pd.notna(model):
        parts.append(str(model))
    return "\n".join(parts)

In [ ]:
# Per-eval-file overview per scanner: manifest metadata, scan size,
# validation coverage, and the target rule (if any). Tables are displayed
# separately, one per scanner.
for run in scanner_runs:
    scans = run["scans"]
    rows = []
    for eval_file, group in scans.groupby("eval_file", dropna=False):
        n = len(group)
        n_validated = int(group["transcript_id"].isin(validated_ids).sum())
        rows.append({
            "eval_file": eval_file,
            "Eval": group["Eval"].iloc[0] if "Eval" in group.columns else None,
            "method": group["method"].iloc[0] if "method" in group.columns else None,
            "model": group["model"].iloc[0] if "model" in group.columns else None,
            "task_set": group["transcript_task_set"].iloc[0],
            "n_scanned": n,
            "n_validated": n_validated,
            "expected_v_rate": group["expected_v_rate"].iloc[0] if "expected_v_rate" in group.columns else None,
            "target_rule": _format_rule(TARGET_RULES.get(eval_file)),
        })

    overview = pd.DataFrame(rows).sort_values(
        ["Eval", "method", "model", "eval_file"], na_position="last"
    )
    run["overview"] = overview
    display(Markdown(f"### {run['label']} — `{run['scanner_key']}`"))
    display(overview)

## Grade Distribution

Stacked grade distribution per benchmark × method × model (aggregating across repeated eval-file runs of the same combination). The two scanners are drawn side by side as subplots.

In [ ]:
score_colors = {0: "#4393c3", 1: "#f4d35e", 2: "#d1495b", 3: "#7f0000"}


def _grade_distribution_data(scans: pd.DataFrame):
    group_cols = ["benchmark", "method"] + (["model"] if "model" in scans.columns else [])
    group_keys = (
        scans.dropna(subset=["benchmark"])
        .groupby(group_cols, dropna=False)
        .size()
        .sort_index()
        .index.tolist()
    )

    def _group_mask(key):
        if not isinstance(key, tuple):
            key = (key,)
        m = pd.Series(True, index=scans.index)
        for col, val in zip(group_cols, key):
            col_series = scans[col]
            if pd.isna(val):
                m &= col_series.isna()
            else:
                m &= (col_series == val)
        return m

    def _group_label(key):
        if not isinstance(key, tuple):
            key = (key,)
        parts = []
        for col, val in zip(group_cols, key):
            if pd.isna(val):
                parts.append(f"(no {col})")
            else:
                parts.append(str(val))
        return "\n".join(parts)

    labels = [_group_label(k) for k in group_keys]
    sample_counts = [int(scans[_group_mask(k)]["value_num"].dropna().shape[0]) for k in group_keys]
    grade_levels = sorted(scans["value_num"].dropna().astype(int).unique())

    stacks = {}
    for grade in grade_levels:
        proportions = []
        for k in group_keys:
            subset_data = scans[_group_mask(k)]["value_num"].dropna()
            total = len(subset_data)
            proportions.append((subset_data.astype(int) == grade).sum() / total if total else 0)
        stacks[grade] = proportions

    return labels, sample_counts, grade_levels, stacks


# Union of grade levels so both axes can share a consistent legend.
all_grade_levels = sorted(set().union(*[
    set(run["scans"]["value_num"].dropna().astype(int).unique())
    for run in scanner_runs
]))


def _group_count(scans: pd.DataFrame) -> int:
    cols = ["benchmark", "method"] + (["model"] if "model" in scans.columns else [])
    return len(scans.dropna(subset=["benchmark"]).groupby(cols, dropna=False))


max_groups = max(_group_count(run["scans"]) for run in scanner_runs)

fig, axes = plt.subplots(
    1, 2,
    figsize=(max(6, max_groups * 1.5) * 2, 5.5),
    sharey=True,
)

for ax, run in zip(axes, scanner_runs):
    labels, sample_counts, _grades, stacks = _grade_distribution_data(run["scans"])
    x = np.arange(len(labels))
    bottom = np.zeros(len(labels))
    for grade in reversed(all_grade_levels):
        proportions = stacks.get(grade, [0] * len(labels))
        ax.bar(x, proportions, bottom=bottom, label=str(grade),
               color=score_colors.get(grade, "#999999"))
        bottom += np.array(proportions)
    for xi, n in zip(x, sample_counts):
        ax.text(xi, .9, f"n={n}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{run['label']} — {run['scanner_key']}")

axes[0].set_ylabel("Proportion")
# Single shared legend drawn on the right-most axis to avoid duplication.
axes[-1].legend(title="Grade", loc="center left", bbox_to_anchor=(1.01, 0.5))
fig.suptitle("Grade Distribution", y=1.02)
fig.tight_layout()
plt.show()

## Detected Violation Rate per Eval File

Fraction of transcripts where the scanner score is `>= VIOLATION_THRESHOLD`, plotted for each scanner in a side-by-side subplot. The reference line shows the target rate from `TARGET_RULES` (validation-mode rules use the mean positive rate in the matched validation rows). Tables below list the same numbers separately per scanner.

In [ ]:
def _expected_rate(eval_file: str, group: pd.DataFrame) -> float | None:
    rule = TARGET_RULES.get(eval_file)
    if rule is None:
        return None
    if rule["mode"] == "uniform":
        return float(rule["positive_rate"])
    if rule["mode"] == "validation":
        ids = group["transcript_id"]
        matched = validation[validation["id"].isin(ids)]
        if matched.empty:
            return None
        return float(matched["target_num"].ge(VIOLATION_THRESHOLD).mean())
    return None


def _violation_df(scans: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for ef in sorted(scans["eval_file"].dropna().unique()):
        group = scans[scans["eval_file"] == ef]
        scores = pd.to_numeric(group["value_num"], errors="coerce")
        n = int(scores.notna().sum())
        v_rate = float(scores.ge(VIOLATION_THRESHOLD).mean()) if n else np.nan
        rows.append({
            "eval_file": ef,
            "label": _short_label(group.iloc[0]),
            "n": n,
            "detected_rate": v_rate,
            "expected_rate": _expected_rate(ef, group),
        })
    return pd.DataFrame(rows)


violation_dfs = [_violation_df(run["scans"]) for run in scanner_runs]
for run, vdf in zip(scanner_runs, violation_dfs):
    run["violation_df"] = vdf

max_files = max(len(vdf) for vdf in violation_dfs)

fig, axes = plt.subplots(
    1, 2,
    figsize=(max(6, max_files * 1.2) * 2, 5.5),
    sharey=True,
)

for ax, run, violation_df in zip(axes, scanner_runs, violation_dfs):
    xs = np.arange(len(violation_df))
    bars = ax.bar(xs, violation_df["detected_rate"], color="#d1495b", label="Detected")
    exp_mask = violation_df["expected_rate"].notna()
    if exp_mask.any():
        ax.scatter(
            xs[exp_mask.values],
            violation_df.loc[exp_mask, "expected_rate"],
            marker="_", s=200, linewidths=3, color="#333333", label="Target",
        )
    for bar, n in zip(bars, violation_df["n"]):
        ax.text(bar.get_x() + bar.get_width() / 2, 0.02, f"n={n}",
                ha="center", va="bottom", fontsize=8, color="white")
    ax.set_xticks(xs)
    ax.set_xticklabels(violation_df["label"], rotation=45, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{run['label']} — {run['scanner_key']}")
    ax.legend(loc="best")

axes[0].set_ylabel(f"Violation rate (score ≥ {VIOLATION_THRESHOLD})")
fig.suptitle("Detected Violation Rate", y=1.02)
fig.tight_layout()
plt.show()

for run, violation_df in zip(scanner_runs, violation_dfs):
    display(Markdown(f"### {run['label']} — `{run['scanner_key']}`"))
    display(violation_df[["label", "n", "detected_rate", "expected_rate"]])

## Performance vs Target Rules

For eval files listed in `TARGET_RULES`, computes accuracy, sensitivity (recall of positives) and specificity (recall of negatives) at the chosen violation threshold. A separate metrics table is displayed per scanner.

In [ ]:
def _build_comparison(scans: pd.DataFrame) -> pd.DataFrame:
    agg_kwargs = dict(
        scanner_grade=("value_num", "first"),
        Eval=("Eval", "first"),
        method=("method", "first"),
        benchmark=("benchmark", "first"),
    )
    if "model" in scans.columns:
        agg_kwargs["model"] = ("model", "first")

    comparison = (
        scans.groupby(["eval_file", "transcript_id"], dropna=False)
        .agg(**agg_kwargs)
        .reset_index()
    )

    scanner_scores = pd.to_numeric(comparison["scanner_grade"], errors="coerce")
    comparison["prediction"] = pd.Series(
        np.where(scanner_scores.isna(), pd.NA, scanner_scores.ge(VIOLATION_THRESHOLD)),
        index=comparison.index,
        dtype="boolean",
    )
    comparison["target"] = pd.Series(pd.NA, index=comparison.index, dtype="boolean")
    comparison["target_grade"] = pd.NA
    comparison["target_source"] = pd.NA

    for eval_file, rule in TARGET_RULES.items():
        mask = comparison["eval_file"] == eval_file
        if not mask.any():
            continue
        mode = rule.get("mode")
        if mode == "uniform":
            rate = rule.get("positive_rate")
            if rate not in {0, 0.0, 1, 1.0}:
                raise ValueError(
                    f"Uniform rules only support 0.0 or 1.0 positive_rate; got {rate!r} for {eval_file}."
                )
            comparison.loc[mask, "target"] = bool(rate)
            comparison.loc[mask, "target_source"] = f"uniform::{rate}"
        elif mode == "validation":
            idx = comparison.index[mask]
            target_grade = comparison.loc[idx, "transcript_id"].map(validation_lookup)
            comparison.loc[idx, "target_grade"] = target_grade.values
            comparison.loc[idx, "target"] = pd.Series(
                np.where(target_grade.isna(), pd.NA, target_grade.ge(VIOLATION_THRESHOLD)),
                index=idx,
                dtype="boolean",
            )
            comparison.loc[idx, "target_source"] = "validation"
        else:
            raise ValueError(f"Unsupported rule mode {mode!r} for {eval_file}")

    return comparison


for run in scanner_runs:
    run["comparison"] = _build_comparison(run["scans"])
    run["valid"] = run["comparison"][
        run["comparison"]["target"].notna() & run["comparison"]["prediction"].notna()
    ].copy()


for run in scanner_runs:
    valid = run["valid"]
    display(Markdown(f"### {run['label']} — `{run['scanner_key']}`"))
    if valid.empty:
        print("No transcript has both a scanner grade and a resolved target — check TARGET_RULES.")
        continue
    metrics_group_cols = ["benchmark", "method"] + (["model"] if "model" in valid.columns else [])
    metrics_rows = []
    for key, group in valid.groupby(metrics_group_cols, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        key_dict = dict(zip(metrics_group_cols, key))
        pred = group["prediction"].astype(bool)
        tgt = group["target"].astype(bool)
        tp = int((pred & tgt).sum())
        tn = int((~pred & ~tgt).sum())
        fp = int((pred & ~tgt).sum())
        fn = int((~pred & tgt).sum())
        n = len(group)
        sources = sorted(group["target_source"].dropna().unique())
        row = {
            "benchmark": key_dict.get("benchmark"),
            "method": key_dict["method"] if pd.notna(key_dict.get("method")) else "—",
        }
        if "model" in metrics_group_cols:
            row["model"] = key_dict["model"] if pd.notna(key_dict.get("model")) else "—"
        row.update({
            "target_source": ", ".join(sources) if sources else "—",
            "n": n,
            "accuracy": (tp + tn) / n if n else np.nan,
            "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        })
        metrics_rows.append(row)
    sort_cols = ["benchmark", "method"] + (["model"] if "model" in metrics_group_cols else [])
    metrics = pd.DataFrame(metrics_rows).sort_values(sort_cols, na_position="last")
    run["metrics"] = metrics
    display_metrics = metrics.copy()
    for col in ["accuracy", "sensitivity", "specificity"]:
        display_metrics[col] = display_metrics[col].map(
            lambda v: f"{v:.1%}" if pd.notna(v) else "—"
        )
    display(display_metrics)

## Disagreements

Per eval file, lists the false-positive and false-negative transcripts so you can open them for inspection. Rendered separately per scanner.

In [ ]:
for run in scanner_runs:
    display(Markdown(f"### {run['label']} — `{run['scanner_key']}`"))
    valid = run["valid"]
    scans = run["scans"]
    if valid.empty:
        print("No data with resolved targets — see above.")
        continue

    detail_cols = ["transcript_id", "scanner_grade", "target_grade",
                   "prediction", "target", "target_source"]
    detail_cols = [c for c in detail_cols if c in valid.columns]

    for eval_file, group in valid.groupby("eval_file", dropna=False):
        label_row = scans[scans["eval_file"] == eval_file].iloc[0]
        label = _short_label(label_row).replace("\n", " / ")
        mismatches = group[group["prediction"].astype(bool) != group["target"].astype(bool)]

        print("=" * 70)
        print(f"{label}")
        print(f"  {eval_file}")
        print(f"  mismatches: {len(mismatches)} / {len(group)} "
              f"({(len(mismatches) / len(group)):.1%})")

        if mismatches.empty:
            print("  perfect agreement")
            continue

        fp = mismatches[mismatches["prediction"].astype(bool)]
        fn = mismatches[~mismatches["prediction"].astype(bool)]
        print(f"  false positives (flagged, target clean): {len(fp)}")
        if not fp.empty:
            display(fp[detail_cols].reset_index(drop=True))
        print(f"  false negatives (missed, target violation): {len(fn)}")
        if not fn.empty:
            display(fn[detail_cols].reset_index(drop=True))

## Confusion Matrices vs Human Labels

For each `(benchmark, model)` combination whose target rule is `validation` (human-labeled), plots a 4x4 confusion matrix (rows = human grade, cols = scanner grade) over grades 0–3 for both scanners side by side, and reports the quadratic-weighted Cohen's kappa. The kappa table below is kept separate per scanner.

In [ ]:
GRADE_LEVELS = [0, 1, 2, 3]


def _confusion_matrix(human: np.ndarray, scanner: np.ndarray, levels: list[int]) -> np.ndarray:
    """Fixed-size confusion matrix; rows = human, cols = scanner."""
    idx = {g: i for i, g in enumerate(levels)}
    k = len(levels)
    cm = np.zeros((k, k), dtype=int)
    for h, s in zip(human, scanner):
        if h in idx and s in idx:
            cm[idx[h], idx[s]] += 1
    return cm


def _quadratic_weighted_kappa(cm: np.ndarray) -> float:
    """Quadratic-weighted Cohen's kappa from a square confusion matrix."""
    n = cm.sum()
    if n == 0:
        return float("nan")
    k = cm.shape[0]
    if k < 2:
        return float("nan")
    weights = (np.arange(k)[:, None] - np.arange(k)[None, :]) ** 2 / (k - 1) ** 2
    observed = cm / n
    row_marg = cm.sum(axis=1) / n
    col_marg = cm.sum(axis=0) / n
    expected = np.outer(row_marg, col_marg)
    denom = (weights * expected).sum()
    if denom == 0:
        return float("nan")
    return 1.0 - (weights * observed).sum() / denom


def _validation_panel_keys(run: dict) -> list[tuple]:
    """Return sorted list of (benchmark[, model]) keys for validation-mode runs."""
    scans = run["scans"]
    files = [
        ef for ef, rule in TARGET_RULES.items()
        if rule.get("mode") == "validation" and (scans["eval_file"] == ef).any()
    ]
    if not files:
        return []
    cols = ["benchmark"] + (["model"] if "model" in scans.columns else [])
    keys = (
        scans[scans["eval_file"].isin(files)]
        .dropna(subset=["benchmark"])
        .groupby(cols, dropna=False)
        .size()
        .sort_index()
        .index.tolist()
    )
    return [k if isinstance(k, tuple) else (k,) for k in keys]


# Use the union of (benchmark, model) keys across both scanners so rows line
# up vertically even if a key is missing from one run.
_panel_cols = ["benchmark"] + (
    ["model"] if any("model" in run["scans"].columns for run in scanner_runs) else []
)
all_panel_keys = sorted(
    set().union(*[set(_validation_panel_keys(run)) for run in scanner_runs])
)


def _key_label(key: tuple) -> str:
    return " / ".join(
        str(v) if not pd.isna(v) else f"(no {col})"
        for col, v in zip(_panel_cols, key)
    )


if not all_panel_keys:
    print("No eval files with validation-mode targets are present in either scan.")
else:
    n_panels = len(all_panel_keys)
    fig, axes = plt.subplots(
        n_panels, 2,
        figsize=(4.2 * 2, 4.0 * n_panels),
        squeeze=False,
    )

    kappa_rows_per_run: list[list[dict]] = [[] for _ in scanner_runs]

    for col_idx, run in enumerate(scanner_runs):
        scans = run["scans"]
        comparison = run["comparison"]
        validation_files = [
            ef for ef, rule in TARGET_RULES.items()
            if rule.get("mode") == "validation" and (scans["eval_file"] == ef).any()
        ]
        eval_to_benchmark = (
            scans.drop_duplicates("eval_file")
            .set_index("eval_file")["benchmark"]
            .to_dict()
        )
        validation_comparison = comparison[comparison["eval_file"].isin(validation_files)].copy()
        validation_comparison["benchmark"] = validation_comparison["eval_file"].map(eval_to_benchmark)

        for row_idx, key in enumerate(all_panel_keys):
            ax = axes[row_idx][col_idx]
            mask = pd.Series(True, index=validation_comparison.index)
            for col, val in zip(_panel_cols, key):
                if col not in validation_comparison.columns:
                    mask &= False
                    continue
                col_series = validation_comparison[col]
                if pd.isna(val):
                    mask &= col_series.isna()
                else:
                    mask &= (col_series == val)
            group = validation_comparison[mask].copy()
            label = _key_label(key)
            if group.empty:
                ax.axis("off")
                ax.set_title(f"{label}\n(no data)", fontsize=9)
                continue
            group["human_grade"] = pd.to_numeric(group["target_grade"], errors="coerce")
            group["scanner_grade_num"] = pd.to_numeric(group["scanner_grade"], errors="coerce")
            paired = group.dropna(subset=["human_grade", "scanner_grade_num"])

            human = paired["human_grade"].astype(int).to_numpy()
            scanner = paired["scanner_grade_num"].astype(int).to_numpy()
            cm = _confusion_matrix(human, scanner, GRADE_LEVELS)
            kappa = _quadratic_weighted_kappa(cm)

            ax.imshow(cm, cmap="Blues", vmin=0)
            ax.set_xticks(range(len(GRADE_LEVELS)))
            ax.set_yticks(range(len(GRADE_LEVELS)))
            ax.set_xticklabels(GRADE_LEVELS)
            ax.set_yticklabels(GRADE_LEVELS)
            ax.set_xlabel("Scanner grade")
            ax.set_ylabel("Human grade")
            vmax = cm.max() if cm.max() > 0 else 1
            for i in range(len(GRADE_LEVELS)):
                for j in range(len(GRADE_LEVELS)):
                    ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                            color="white" if cm[i, j] > vmax / 2 else "black", fontsize=9)
            kappa_str = f"{kappa:.3f}" if not np.isnan(kappa) else "—"
            ax.set_title(
                f"{run['label']} — {label}\nn={len(paired)}, qwκ={kappa_str}",
                fontsize=9,
            )
            row_out = {col: (val if not pd.isna(val) else None)
                       for col, val in zip(_panel_cols, key)}
            row_out.update({
                "n": len(paired),
                "quadratic_weighted_kappa": kappa,
            })
            kappa_rows_per_run[col_idx].append(row_out)

    fig.suptitle("Confusion Matrices vs Human Labels", y=1.0)
    fig.tight_layout()
    plt.show()

    for run, kappa_rows in zip(scanner_runs, kappa_rows_per_run):
        display(Markdown(f"### {run['label']} — `{run['scanner_key']}`"))
        kappa_df = pd.DataFrame(kappa_rows)
        run["kappa_df"] = kappa_df
        if kappa_df.empty:
            print("No validation-mode benchmarks present for this scanner.")
            continue
        display_kappa = kappa_df.copy()
        display_kappa["quadratic_weighted_kappa"] = display_kappa["quadratic_weighted_kappa"].map(
            lambda v: f"{v:.3f}" if pd.notna(v) else "—"
        )
        display(display_kappa)

## Combined Across All Evals

Aggregates across every eval file with a resolved target. Per-scanner metrics tables are kept separate; the pooled confusion matrices are drawn side by side.

In [ ]:
# Pooled metrics per scanner (tables kept separate).
for run in scanner_runs:
    display(Markdown(f"### {run['label']} — `{run['scanner_key']}`"))
    valid = run["valid"]
    if valid.empty:
        print("No data with resolved targets — nothing to aggregate.")
        continue
    pred_all = valid["prediction"].astype(bool)
    tgt_all = valid["target"].astype(bool)
    tp = int((pred_all & tgt_all).sum())
    tn = int((~pred_all & ~tgt_all).sum())
    fp = int((pred_all & ~tgt_all).sum())
    fn = int((~pred_all & tgt_all).sum())
    n = len(valid)
    sources = sorted(valid["target_source"].dropna().unique())
    combined_metrics = pd.DataFrame([{
        "scope": "all evals combined",
        "target_source": ", ".join(sources) if sources else "—",
        "n": n,
        "accuracy": (tp + tn) / n if n else np.nan,
        "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }])
    run["combined_metrics"] = combined_metrics
    display_combined = combined_metrics.copy()
    for col in ["accuracy", "sensitivity", "specificity"]:
        display_combined[col] = display_combined[col].map(
            lambda v: f"{v:.1%}" if pd.notna(v) else "—"
        )
    display(display_combined)

# Pooled confusion matrices across validation-mode rows, plotted side by side.
pooled_paired = []
for run in scanner_runs:
    valid = run["valid"]
    if valid.empty:
        pooled_paired.append(None)
        continue
    validation_rows = valid[valid["target_source"] == "validation"].copy()
    validation_rows["human_grade"] = pd.to_numeric(validation_rows["target_grade"], errors="coerce")
    validation_rows["scanner_grade_num"] = pd.to_numeric(validation_rows["scanner_grade"], errors="coerce")
    paired = validation_rows.dropna(subset=["human_grade", "scanner_grade_num"])
    pooled_paired.append(paired if not paired.empty else None)

if all(p is None for p in pooled_paired):
    print("No validation-mode rows in either scanner — skipping combined confusion matrix.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(4.2 * 2, 4.4))
    for ax, run, paired in zip(axes, scanner_runs, pooled_paired):
        if paired is None:
            ax.axis("off")
            ax.set_title(f"{run['label']}\n(no validation-mode rows)", fontsize=10)
            continue
        human = paired["human_grade"].astype(int).to_numpy()
        scanner = paired["scanner_grade_num"].astype(int).to_numpy()
        cm = _confusion_matrix(human, scanner, GRADE_LEVELS)
        kappa = _quadratic_weighted_kappa(cm)

        ax.imshow(cm, cmap="Blues", vmin=0)
        ax.set_xticks(range(len(GRADE_LEVELS)))
        ax.set_yticks(range(len(GRADE_LEVELS)))
        ax.set_xticklabels(GRADE_LEVELS)
        ax.set_yticklabels(GRADE_LEVELS)
        ax.set_xlabel("Scanner grade")
        ax.set_ylabel("Human grade")
        vmax = cm.max() if cm.max() > 0 else 1
        for i in range(len(GRADE_LEVELS)):
            for j in range(len(GRADE_LEVELS)):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > vmax / 2 else "black", fontsize=9)
        kappa_str = f"{kappa:.3f}" if not np.isnan(kappa) else "—"
        ax.set_title(
            f"{run['label']}\nn={len(paired)}, qwκ={kappa_str}", fontsize=10
        )

    fig.suptitle("All validation evals combined", y=0.99)
    fig.tight_layout()
    plt.show()

## Scanner vs Scanner Agreement

Compares the two scanners directly against each other (independent of human labels). Transcripts are paired on `(eval_file, transcript_id)` and only kept when both scanners produced a numeric grade. Reports a 4x4 confusion matrix with quadratic-weighted Cohen's kappa, an overall disagreement summary, and a per-`(benchmark, model)` breakdown of where the scanners diverge.

In [ ]:
run_a, run_b = scanner_runs[0], scanner_runs[1]
label_a, label_b = run_a["label"], run_b["label"]


def _scanner_pair_frame(run: dict, suffix: str) -> pd.DataFrame:
    agg = {
        f"grade_{suffix}": ("value_num", "first"),
        "Eval": ("Eval", "first"),
        "method": ("method", "first"),
        "benchmark": ("benchmark", "first"),
    }
    if "model" in run["scans"].columns:
        agg["model"] = ("model", "first")
    df = (
        run["scans"]
        .groupby(["eval_file", "transcript_id"], dropna=False)
        .agg(**agg)
        .reset_index()
    )
    df[f"grade_{suffix}"] = pd.to_numeric(df[f"grade_{suffix}"], errors="coerce")
    return df


pair_a = _scanner_pair_frame(run_a, "a")
pair_b = _scanner_pair_frame(run_b, "b")[["eval_file", "transcript_id", "grade_b"]]

paired = pair_a.merge(pair_b, on=["eval_file", "transcript_id"], how="inner")
paired_valid = paired.dropna(subset=["grade_a", "grade_b"]).copy()
paired_valid["grade_a"] = paired_valid["grade_a"].astype(int)
paired_valid["grade_b"] = paired_valid["grade_b"].astype(int)

print(f"Transcripts scored by both scanners: {len(paired_valid):,} "
      f"(of {len(paired):,} paired rows)")

if paired_valid.empty:
    print("No overlap between the two scanners — nothing to compare.")
else:
    cm = _confusion_matrix(
        paired_valid["grade_a"].to_numpy(),
        paired_valid["grade_b"].to_numpy(),
        GRADE_LEVELS,
    )
    kappa = _quadratic_weighted_kappa(cm)
    agree = int(np.trace(cm))
    n = int(cm.sum())
    off_by_one = int(
        sum(cm[i, j] for i in range(len(GRADE_LEVELS))
            for j in range(len(GRADE_LEVELS)) if abs(i - j) == 1)
    )

    fig, ax = plt.subplots(figsize=(4.6, 4.4))
    ax.imshow(cm, cmap="Blues", vmin=0)
    ax.set_xticks(range(len(GRADE_LEVELS)))
    ax.set_yticks(range(len(GRADE_LEVELS)))
    ax.set_xticklabels(GRADE_LEVELS)
    ax.set_yticklabels(GRADE_LEVELS)
    ax.set_xlabel(f"{label_b} grade")
    ax.set_ylabel(f"{label_a} grade")
    vmax = cm.max() if cm.max() > 0 else 1
    for i in range(len(GRADE_LEVELS)):
        for j in range(len(GRADE_LEVELS)):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > vmax / 2 else "black", fontsize=10)
    kappa_str = f"{kappa:.3f}" if not np.isnan(kappa) else "—"
    ax.set_title(
        f"{label_a} vs {label_b}\nn={n}, agree={agree} ({agree/n:.1%}), qwκ={kappa_str}"
    )
    fig.tight_layout()
    plt.show()

    display(Markdown("### Disagreement summary"))
    disagree_mask = paired_valid["grade_a"] != paired_valid["grade_b"]
    a_higher = int((paired_valid["grade_a"] > paired_valid["grade_b"]).sum())
    b_higher = int((paired_valid["grade_b"] > paired_valid["grade_a"]).sum())
    diffs = (paired_valid["grade_a"] - paired_valid["grade_b"]).abs()
    summary = pd.DataFrame([{
        "n_paired": n,
        "agree": agree,
        "agree_rate": agree / n,
        "disagree": int(disagree_mask.sum()),
        "off_by_one": off_by_one,
        f"{label_a}_higher": a_higher,
        f"{label_b}_higher": b_higher,
        "mean_abs_diff": float(diffs.mean()),
        "max_abs_diff": int(diffs.max()),
        "quadratic_weighted_kappa": kappa,
    }])
    display_summary = summary.copy()
    display_summary["agree_rate"] = display_summary["agree_rate"].map(lambda v: f"{v:.1%}")
    display_summary["mean_abs_diff"] = display_summary["mean_abs_diff"].map(lambda v: f"{v:.2f}")
    display_summary["quadratic_weighted_kappa"] = display_summary["quadratic_weighted_kappa"].map(
        lambda v: f"{v:.3f}" if pd.notna(v) else "—"
    )
    display(display_summary)

    pb_group_cols = ["benchmark"] + (["model"] if "model" in paired_valid.columns else [])
    display(Markdown(f"### Per-{' × '.join(pb_group_cols)} agreement"))
    per_bench_rows = []
    for key, group in paired_valid.groupby(pb_group_cols, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        key_dict = dict(zip(pb_group_cols, key))
        g_a = group["grade_a"].to_numpy()
        g_b = group["grade_b"].to_numpy()
        cm_b = _confusion_matrix(g_a, g_b, GRADE_LEVELS)
        n_b = int(cm_b.sum())
        agree_b = int(np.trace(cm_b))
        diffs_b = np.abs(g_a - g_b)
        row = {
            "benchmark": key_dict.get("benchmark") if pd.notna(key_dict.get("benchmark")) else "—",
        }
        if "model" in pb_group_cols:
            row["model"] = key_dict["model"] if pd.notna(key_dict.get("model")) else "—"
        row.update({
            "n": n_b,
            "agree_rate": agree_b / n_b if n_b else np.nan,
            f"{label_a}_higher": int((g_a > g_b).sum()),
            f"{label_b}_higher": int((g_b > g_a).sum()),
            "mean_abs_diff": float(diffs_b.mean()) if n_b else np.nan,
            "quadratic_weighted_kappa": _quadratic_weighted_kappa(cm_b),
        })
        per_bench_rows.append(row)
    sort_cols = ["benchmark"] + (["model"] if "model" in pb_group_cols else [])
    per_bench = pd.DataFrame(per_bench_rows).sort_values(sort_cols, na_position="last")
    display_pb = per_bench.copy()
    display_pb["agree_rate"] = display_pb["agree_rate"].map(
        lambda v: f"{v:.1%}" if pd.notna(v) else "—"
    )
    display_pb["mean_abs_diff"] = display_pb["mean_abs_diff"].map(
        lambda v: f"{v:.2f}" if pd.notna(v) else "—"
    )
    display_pb["quadratic_weighted_kappa"] = display_pb["quadratic_weighted_kappa"].map(
        lambda v: f"{v:.3f}" if pd.notna(v) else "—"
    )
    display(display_pb)

    display(Markdown("### Largest disagreements"))
    disagreements = paired_valid[disagree_mask].copy()
    disagreements["abs_diff"] = (disagreements["grade_a"] - disagreements["grade_b"]).abs()
    disagreements = disagreements.rename(
        columns={"grade_a": f"{label_a}_grade", "grade_b": f"{label_b}_grade"}
    )
    disagreements = disagreements.sort_values(
        ["abs_diff", "benchmark", "eval_file"], ascending=[False, True, True]
    )
    cols = ["benchmark"] + (["model"] if "model" in disagreements.columns else []) + [
        "method", "eval_file", "transcript_id",
        f"{label_a}_grade", f"{label_b}_grade", "abs_diff",
    ]
    display(disagreements[cols].head(50).reset_index(drop=True))
    if len(disagreements) > 50:
        print(f"(showing top 50 of {len(disagreements)} disagreements)")

## Composite Scanner vs Human Labels

Combine the two scanner grades into a single composite per transcript and evaluate against human labels. Two composite strategies are compared side by side:

- **floor(mean)** — average the two scanner grades and round down to keep an integer score.
- **max** — take the higher of the two scanner grades.

Restricted to transcripts where both scanners produced a numeric grade *and* a human grade is available (i.e. the eval file has a `validation`-mode rule). Reports a 4x4 confusion matrix with quadratic-weighted Cohen's kappa, plus violation-threshold metrics (accuracy, sensitivity, specificity) for each composite.

In [ ]:
# Pair both scanner grades with the human grade per transcript.
composite_base = paired_valid.copy()
composite_base["human_grade"] = composite_base["transcript_id"].map(validation_lookup)

# Restrict to validation-mode eval files so we have a real human label.
_validation_files = {
    ef for ef, rule in TARGET_RULES.items() if rule.get("mode") == "validation"
}
composite_base = composite_base[composite_base["eval_file"].isin(_validation_files)]
composite_base = composite_base.dropna(subset=["human_grade"]).copy()
composite_base["human_grade"] = composite_base["human_grade"].astype(int)

composite_base["composite_floor_mean"] = np.floor(
    (composite_base["grade_a"] + composite_base["grade_b"]) / 2
).astype(int)
composite_base["composite_max"] = np.maximum(
    composite_base["grade_a"], composite_base["grade_b"]
).astype(int)

composites = [
    ("floor(mean)", "composite_floor_mean"),
    ("max", "composite_max"),
]

print(f"Transcripts with both scanner grades and a human label: {len(composite_base):,}")

if composite_base.empty:
    print("No overlap — nothing to score.")
else:
    fig, axes = plt.subplots(1, len(composites), figsize=(4.6 * len(composites), 4.4))
    if len(composites) == 1:
        axes = [axes]

    metrics_rows = []
    for ax, (name, col) in zip(axes, composites):
        human = composite_base["human_grade"].to_numpy()
        comp = composite_base[col].to_numpy()
        cm = _confusion_matrix(human, comp, GRADE_LEVELS)
        kappa = _quadratic_weighted_kappa(cm)

        ax.imshow(cm, cmap="Blues", vmin=0)
        ax.set_xticks(range(len(GRADE_LEVELS)))
        ax.set_yticks(range(len(GRADE_LEVELS)))
        ax.set_xticklabels(GRADE_LEVELS)
        ax.set_yticklabels(GRADE_LEVELS)
        ax.set_xlabel("Composite grade")
        ax.set_ylabel("Human grade")
        vmax = cm.max() if cm.max() > 0 else 1
        for i in range(len(GRADE_LEVELS)):
            for j in range(len(GRADE_LEVELS)):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > vmax / 2 else "black", fontsize=10)
        kappa_str = f"{kappa:.3f}" if not np.isnan(kappa) else "\u2014"
        ax.set_title(f"composite = {name}\nn={len(composite_base)}, qw\u03ba={kappa_str}")

        pred = comp >= VIOLATION_THRESHOLD
        tgt = human >= VIOLATION_THRESHOLD
        tp = int((pred & tgt).sum())
        tn = int((~pred & ~tgt).sum())
        fp = int((pred & ~tgt).sum())
        fn = int((~pred & tgt).sum())
        n = len(composite_base)
        metrics_rows.append({
            "composite": name,
            "n": n,
            "accuracy": (tp + tn) / n if n else np.nan,
            "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "quadratic_weighted_kappa": kappa,
            "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        })

    fig.suptitle(f"Composite of {label_a} + {label_b} vs human grade", y=1.02)
    fig.tight_layout()
    plt.show()

    metrics_df = pd.DataFrame(metrics_rows)
    display_metrics = metrics_df.copy()
    for col in ["accuracy", "sensitivity", "specificity"]:
        display_metrics[col] = display_metrics[col].map(
            lambda v: f"{v:.1%}" if pd.notna(v) else "\u2014"
        )
    display_metrics["quadratic_weighted_kappa"] = display_metrics["quadratic_weighted_kappa"].map(
        lambda v: f"{v:.3f}" if pd.notna(v) else "\u2014"
    )
    display(Markdown(f"### Composite metrics vs human grade (threshold = {VIOLATION_THRESHOLD})"))
    display(display_metrics)

### Performance partitioned by scanner agreement

Splits the composite-eligible transcripts into four mutually-exclusive subsets based on what each individual scanner said (composites are not used here — each row reports a single scanner's call against the human grade):

- **both agreed positive (≥ threshold)** — both scanners flagged. Predictions are the same so a single row covers both.
- **both agreed negative (< threshold)** — both scanners cleared. Single row.
- **disagreed, `{label_a}` positive** — only scanner A flagged. Row reports A's prediction (positive) vs human.
- **disagreed, `{label_b}` positive** — only scanner B flagged. Row reports B's prediction (positive) vs human.

Within each disagreement row, the listed scanner's prediction is True for every transcript by construction, so accuracy reduces to that scanner's precision in the disagreement region — i.e. *when this scanner is the only one calling violation, how often is it right?*

In [ ]:
if composite_base.empty:
    print("No data \u2014 see composite cell above.")
else:
    a_pos = (composite_base["grade_a"] >= VIOLATION_THRESHOLD).to_numpy()
    b_pos = (composite_base["grade_b"] >= VIOLATION_THRESHOLD).to_numpy()
    tgt_all = (composite_base["human_grade"].to_numpy() >= VIOLATION_THRESHOLD)

    # (label, mask, scanner_label, prediction_array)
    partitions = [
        ("both agreed positive (\u2265 threshold)", a_pos & b_pos, "both", a_pos & b_pos),
        ("both agreed negative (< threshold)", (~a_pos) & (~b_pos), "both", a_pos & b_pos),
        (f"disagreed, {label_a} positive", a_pos & ~b_pos, label_a, a_pos),
        (f"disagreed, {label_b} positive", b_pos & ~a_pos, label_b, b_pos),
    ]

    rows = []
    for subset_name, mask, scanner_label, pred_full in partitions:
        n = int(mask.sum())
        if n == 0:
            rows.append({
                "subset": subset_name,
                "scanner": scanner_label,
                "n": 0,
                "human_violation_rate": np.nan,
                "accuracy": np.nan,
                "sensitivity": np.nan,
                "specificity": np.nan,
                "tp": 0, "fp": 0, "tn": 0, "fn": 0,
            })
            continue
        tgt = tgt_all[mask]
        pred = pred_full[mask]
        tp = int((pred & tgt).sum())
        tn = int((~pred & ~tgt).sum())
        fp = int((pred & ~tgt).sum())
        fn = int((~pred & tgt).sum())
        rows.append({
            "subset": subset_name,
            "scanner": scanner_label,
            "n": n,
            "human_violation_rate": float(tgt.mean()),
            "accuracy": (tp + tn) / n,
            "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        })

    agreement_metrics = pd.DataFrame(rows)
    display(Markdown(f"### Accuracy by scanner agreement (threshold = {VIOLATION_THRESHOLD})"))
    display_agree = agreement_metrics.copy()
    for col in ["human_violation_rate", "accuracy", "sensitivity", "specificity"]:
        display_agree[col] = display_agree[col].map(
            lambda v: f"{v:.1%}" if pd.notna(v) else "\u2014"
        )
    display(display_agree)

    # Aggregate across the agreement region and the disagreement region.
    # In the agreement region every prediction is the agreed-upon call (no
    # composite needed). In the disagreement region we apply each composite
    # strategy from the previous section to resolve a single prediction per
    # transcript.
    agree_mask = (a_pos & b_pos) | ((~a_pos) & (~b_pos))
    disagree_mask = a_pos ^ b_pos

    agg_rows = []

    # Agreement: pred = whatever both scanners agreed on (== a_pos in this region).
    if agree_mask.any():
        tgt = tgt_all[agree_mask]
        pred = a_pos[agree_mask]
        tp = int((pred & tgt).sum())
        tn = int((~pred & ~tgt).sum())
        fp = int((pred & ~tgt).sum())
        fn = int((~pred & tgt).sum())
        n = int(agree_mask.sum())
        agg_rows.append({
            "subset": "agreement",
            "composite": "\u2014",
            "n": n,
            "human_violation_rate": float(tgt.mean()),
            "accuracy": (tp + tn) / n,
            "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        })
    else:
        agg_rows.append({
            "subset": "agreement", "composite": "\u2014", "n": 0,
            "human_violation_rate": np.nan, "accuracy": np.nan,
            "sensitivity": np.nan, "specificity": np.nan,
            "tp": 0, "fp": 0, "tn": 0, "fn": 0,
        })

    # Disagreement: one row per composite strategy.
    for name, col in composites:
        if disagree_mask.any():
            tgt = tgt_all[disagree_mask]
            pred = (composite_base[col].to_numpy()[disagree_mask]) >= VIOLATION_THRESHOLD
            tp = int((pred & tgt).sum())
            tn = int((~pred & ~tgt).sum())
            fp = int((pred & ~tgt).sum())
            fn = int((~pred & tgt).sum())
            n = int(disagree_mask.sum())
            agg_rows.append({
                "subset": "disagreement",
                "composite": name,
                "n": n,
                "human_violation_rate": float(tgt.mean()),
                "accuracy": (tp + tn) / n,
                "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
                "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
                "tp": tp, "fp": fp, "tn": tn, "fn": fn,
            })
        else:
            agg_rows.append({
                "subset": "disagreement", "composite": name, "n": 0,
                "human_violation_rate": np.nan, "accuracy": np.nan,
                "sensitivity": np.nan, "specificity": np.nan,
                "tp": 0, "fp": 0, "tn": 0, "fn": 0,
            })

    aggregate_metrics = pd.DataFrame(agg_rows)
    display(Markdown("### Aggregated across agreement vs disagreement"))
    display_aggregate = aggregate_metrics.copy()
    for col in ["human_violation_rate", "accuracy", "sensitivity", "specificity"]:
        display_aggregate[col] = display_aggregate[col].map(
            lambda v: f"{v:.1%}" if pd.notna(v) else "\u2014"
        )
    display(display_aggregate)